# Prompt engineering approach for Named Entity Recognition (NER) in Ukrainian texts using large language models

Choosen setup: 👉 Ollama + Llama-3.2-3BLlama-3.2-3B

In [66]:
from numpy.random import randint

In [1]:
from numpy.random import randint
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model='llama3.2', messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
])
print(response['message']['content'])
# or access fields directly from the response object
print(response.message.content)

The sky appears blue because of a phenomenon called scattering, which occurs when sunlight interacts with the tiny molecules of gases in the Earth's atmosphere.

Here's what happens:

1. Sunlight enters the Earth's atmosphere and is composed of a spectrum of colors, including all the colors of the rainbow.
2. The shorter (blue) wavelengths of light are scattered more than the longer (red) wavelengths by the small molecules of gases such as nitrogen (N2) and oxygen (O2).
3. This scattering effect is more pronounced when the sunlight hits the atmospheric particles at a shallow angle, which happens during the daytime when the sun is overhead.
4. As a result, the blue light is dispersed in all directions, reaching our eyes from every part of the sky.
5. Our brains interpret this scattered blue light as the color of the sky.

The reason why the sky doesn't appear violet (the color of light with the shortest wavelength) is that there are more nitrogen molecules than oxygen molecules in the a

### Data loading and review

In [2]:
%load_ext autoreload
%autoreload 2

# For importing inner modules
import sys
from pathlib import Path

root = Path().resolve().parent 
sys.path.append(str(root))


In [9]:
import pandas as pd

train = pd.read_csv("../data/train.csv")
train.head()

,id,text,entities
0,582dab45ea473d07323e3c6a0415425c037e63b5a7ab0a...,"Зрозуміло , що український бізнес почав викори...","[{""label"": ""MISC"", ""text"": ""\u041a\u0421\u0412..."
1,df0933559918588668cb4219c80908b33cdef041dce468...,\nДолучилися до заходу заступник голови райдер...,"[{""label"": ""JOB"", ""text"": ""\u0437\u0430\u0441\..."
2,6b66cabd04bce970de47e4dbef5c1c0219591d296d4285...,"У неділю , 4 березня , у Чернівцях у ліцеї № 4...","[{""label"": ""DATE"", ""text"": ""4 \u0431\u0435\u04..."
3,3ca90ad4d912c780b305407035af5450c5136d52d11497...,"Міжнародний благодійний фонд « Покуття » , фун...","[{""label"": ""ORG"", ""text"": ""\u041c\u0456\u0436\..."
4,27a06d92f27fdb30ffc4a13cafd98d81ae1cc7f436b865...,"Цю прогалину , життєпис відомого колись на всю...","[{""label"": ""JOB"", ""text"": ""\u0444\u043e\u0442\..."


In [8]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        391 non-null    object
 1   text      391 non-null    object
 2   entities  391 non-null    object
dtypes: object(3)
memory usage: 9.3+ KB


In [14]:
def run_ner(text, template):
    # Insert the text into the prompt template
    prompt = template.replace("{{TEXT_HERE}}", text)

    # Query the local Ollama model
    response: ChatResponse = chat(
        model="llama3.2",   # or "llama3.2:3b", depending on your local model name
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response["message"]["content"]

In [9]:
with open("../prompts/ner-prompt.txt", "r", encoding="utf-8") as f:
    template = f.read()

# Example usage
text=train.text[0]

output = run_ner(text, template)
print(output)

[
  {"label": "PERS", "text": "Фармак"},
  {"label": "PERIOD", "text": "2018 рік"},
  {"label": "LOC", "text": "Київ"},
  {"label": "ORG", "text": "Фармак"},
  {"label": "DOC", "text": "Глобальний договір ООН"},
  {"label": "PERIOD", "text": "2014 рік"},
  {"label": "MISC", "text": "Життєлюб Гаріка Корогодського"},
  {"label": "ORG", "text": "Рошен"},
  {"label": "PERIOD", "text": "2014 рік"},
  {"label": "DOC", "text": "Глобальна угруповка Росії"},
  {"label": "PERS", "text": "Рошен"},
  {"label": "ORG", "text": "Німеччина"},
  {"label": "ORG", "text": "Австралія"},
  {"label": "ORG", "text": "Швеція"},
  {"label": "ORG", "text": "США"}
]


In [13]:
from src.evaluation.ner_f1_score import ner_f1_score

print(ner_f1_score(output, train.entities[0]))

{'precision': 0.07142857142857142, 'recall': 0.03571428571428571, 'f1': 0.047619047619047616}


In [20]:
train.text[0]

'Зрозуміло , що український бізнес почав використовувати КСВ як інструмент формування своєї репутації буквально декілька років тому .\nЗ одного боку , саме через це більшість проектів КСВ здійснюються епізодично та деколи виглядають , радше , як просто благодійність .\nВинятком будуть хіба що представництва іноземних корпорацій .\nЗ іншого боку , для українських компаній відкривається потужне « вікно можливостей » , щоб втілювати власні унікальні стратегії .\n\nФілантропія як пріоритет\n\nНайпоширенішим підходом українського бізнесу у КСВ є філантропія — компанії виділяють кошти на суспільно важливі проекти .\nЦе може бути як власна ініціатива , так і відповідь на запит від громади .\nЦе абсолютно нормальна практика через недостатній рівень статків та доходів переважної більшості українських громадян та громад .\nІ хоча деякі місцеві експерти вважають це базовим рівнем , що свідчить про недостатній розвиток КСВ , ця діяльність заслуговує на повагу .\nОсобливо якщо зауважити , що такі п

In [19]:
print(train.entities[0])

[{"label": "MISC", "text": "\u041a\u0421\u0412"}, {"label": "MISC", "text": "\u041a\u0421\u0412"}, {"label": "MISC", "text": "\u041a\u0421\u0412"}, {"label": "MISC", "text": "\u041a\u0421\u0412"}, {"label": "ORG", "text": "\u0416\u0438\u0442\u0442\u0454\u043b\u044e\u0431"}, {"label": "PERS", "text": "\u0413\u0430\u0440\u0456\u043a\u0430 \u041a\u043e\u0440\u043e\u0433\u043e\u0434\u0441\u044c\u043a\u043e\u0433\u043e"}, {"label": "MISC", "text": "\u0413\u043b\u043e\u0431\u0430\u043b\u044c\u043d\u043e\u043c\u0443 \u0434\u043e\u0433\u043e\u0432\u043e\u0440\u0443"}, {"label": "MISC", "text": "\u0426\u0456\u043b\u044f\u043c \u0441\u0442\u0430\u043b\u043e\u0433\u043e \u0440\u043e\u0437\u0432\u0438\u0442\u043a\u0443 \u041e\u041e\u041d"}, {"label": "ORG", "text": "\u0417\u0431\u0440\u043e\u0439\u043d\u0438\u0445 \u0421\u0438\u043b"}, {"label": "MISC", "text": "\u041a\u0421\u0412"}, {"label": "ORG", "text": "\u0417\u0431\u0440\u043e\u0439\u043d\u0456 \u0421\u0438\u043b\u0438 \u0423\u043a\u0440\u0

In [15]:
len(train)

391

In [21]:
from src.evaluation.ner_f1_score import evaluate_dataset

y_pred = []

def run_lmm(row):
    return pd.Series({ 'entities': run_ner(row.text) })

idx = 0
for row in train.itertuples(index=False):
    y_pred.append({ 'id': row.id, 'entities': run_ner(row.text) })
    if idx % 50 == 0:
        print("Progress: ", idx)
    idx += 1

pd.DataFrame(y_pred).to_csv("submission_1.csv", index=False)


Progress:  0
Progress:  50
Progress:  100
Progress:  150
Progress:  200
Progress:  250
Progress:  300
Progress:  350


KeyboardInterrupt: 

In [25]:
pd.DataFrame(y_pred).to_csv("submission_1.csv", index=False)

In [26]:
y_pred = pd.DataFrame(y_pred)

In [30]:
evaluate_dataset(y_pred['entities'], train['entities'])

{'precision': 0.021009827177228057,
 'recall': 0.19935691318327975,
 'f1': 0.03801348865726548}

### Few-shot promptimg

5 rand records from train will be used for examples 

In [10]:
from sklearn.model_selection import train_test_split

train, few_shot_records = train_test_split(train, test_size=5, random_state=42, shuffle=True)

In [11]:
for row in few_shot_records.itertuples(index=False):
    print(row.id)

64fbdbd69e52bd7ffa0fcb7c07ba169b4334e6ec08f3e3bbea62b4dc7b307864
0f95d4e44f750d8b70b316ba59cf16a143709dd24dfef1e9ee1fcc1575430efa
463efc4dcb268d54815b0b32103bc0e2b0f368e6736e97503530a901437d17aa
e5f133f4992502076f1ba9715d0f03f873d83a1ab9c4a8074733964155cf8503
c3f2d64d3feebf4b7cce22ca3aa1b8c22ee986e3f723991b0631dd1bf6e9d1c4


In [21]:
few_shot_records.reset_index(inplace=True)

In [27]:
from src.utils.decode import decode_unicode_escape

decode_unicode_escape(few_shot_records.entities[0])

'[{"label": "QUANT", "text": "4"}, {"label": "QUANT", "text": "6 м"}, {"label": "LOC", "text": "Дніпра"}, {"label": "LOC", "text": "Інгулу"}, {"label": "LOC", "text": "Громоклеї"}, {"label": "JOB", "text": "кошовий отаман"}, {"label": "PERS", "text": "Іван Сірко"}, {"label": "LOC", "text": "Чорному лісі"}, {"label": "LOC", "text": "Чортомлицької січі"}, {"label": "LOC", "text": "Грушівці"}, {"label": "DATE", "text": "1680 році"}, {"label": "LOC", "text": "Очаківській стороні"}]'

In [36]:
from src.utils.decode import decode_unicode_escape

TEXT_HERE = '{{TEXT_HERE}}'
ENTITIES_HERE = '{{ENTITIES_HERE}}'
FEW_SHOT_EXAMPLES_HERE = '{{FEW_SHOT_EXAMPLES_HERE}}'
INDEX_HERE = '{{INDEX_HERE}}'

with open("../prompts/ner-prompt-few-shot.txt", "r", encoding="utf-8") as f:
    few_shot_template = f.read()

with open("../prompts/few-shot-temp.txt", "r", encoding="utf-8") as f:
    few_shot_item_templ = f.read()

def prepare_few_shot_prompt(examples, template, few_shot_item_templ):
    few_shot_prompt = ""
    for row in examples.itertuples():
        few_shot_prompt += few_shot_item_templ.replace(TEXT_HERE, row.text).replace(ENTITIES_HERE, decode_unicode_escape(row.entities)).replace(INDEX_HERE, str(row.index))

    return template.replace(FEW_SHOT_EXAMPLES_HERE, few_shot_prompt)

few_shot_template = prepare_few_shot_prompt(few_shot_records, few_shot_template, few_shot_item_templ)


Quick check

In [38]:
text=train.text[0]

output = run_ner(text, few_shot_template)
print(output)

[{"label": "ТОВ « Фармак »", "text": "випроваджує великі проекти в сфері КСВ за трьома пріоритетними напрямками: екологія, здоровий спосіб життя та корпоративне волонтерство."}, {"label": "Корпорація Рошен", "text": "біла на закупівлі для ЗСУ 700-800 млн грн."}]


In [39]:
text=train.text[0]

output = run_ner(text, few_shot_template)
print(output)

[{"label": "Представництво іноземних корпорацій", "text": " corporation Roshen"}, {"label": "Фармацевтична компанія ", "text": " Фармак "}, {"label": "Компанії що вносять проекти в сфері КСВ ", "text": " Фармак "}]


In [40]:
def run_ner(text, template):
    # Insert the text into the prompt template
    prompt = template.replace("{{TEXT_HERE}}", text)

    # Query the local Ollama model
    response: ChatResponse = chat(
        model="llama3.2",   # or "llama3.2:3b", depending on your local model name
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={"temperature": 0}
    )

    return response["message"]["content"]

In [41]:
output = run_ner(text, few_shot_template)
print(output)

[{"label": "Вищий господарський суд України", "text": "ТОВ «Іллінецький торговий дом»"}, {"label": "QUANT", "text": "10 га"}, {"label": "QUANT", "text": "150 метрів"}, {"label": "LOC", "text": "Чорного моря"}, {"label": "LOC", "text": "Новофедорівкою"}, {"label": "LOC", "text": "Сакському районі Криму"}, {"label": "DATE", "text": "28 травня"}, {"label": "MON", "text": "2,07 млн грн."}, {"label": "ORG", "text": "Державна інспекція сільського господарства в АРК"}, {"label": "ORG", "text": "Сакська міжрайонна прокуратура"}, {"label": "DATE", "text": "10.09.2010"}, {"label": "ORG", "text": "Сакською райдержадміністрацією"}, {"label": "ORG", "text": "ТОВ «Іллінецький торговий дом»"}, {"label": "ORG", "text": "господарського суду АРК"}, {"label": "DATE", "text": "05.07.2010"}, {"label": "DATE", "text": "січні цього року"}, {"label": "ORG", "text": "Севастопольським апеляційним Госпсудом"}, {"label": "JOB", "text": "прокурор"}, {"label": "ORG", "text": "Держінспекція сільського господарства"}

In [42]:
def run_lmm(row, template):
    return pd.Series({ 'entities': run_ner(row.text, template) })

def ner_for_df(df: pd.DataFrame, template):
    y_pred = []

    idx = 0
    for row in df.itertuples(index=False):
        y_pred.append({ 'id': row.id, 'entities': run_ner(row.text, template) })
        if idx % 50 == 0:
            print("Progress: ", idx)
        idx += 1

    return y_pred


In [46]:
y_pred = []

idx = 0
for row in train.head(10).itertuples(index=False):
    y_pred.append({ 'id': row.id, 'entities': run_ner(row.text, few_shot_template) })
    print("Progress: ", idx)
    idx += 1

Progress:  0
Progress:  1


KeyboardInterrupt: 

In [47]:
y_pred

[{'id': '83615dbd8e86179e34fa2e002cdd62216e07568947a50d65ad059eb19fba17d2',
  'entities': '[{"label": "Вищий господарський суд України", "text": "Роман Сміяненко став в. о. начальника Служби автомобільних доріг у Чернігівській області 16 грудня 2019 р."}, {"label": "ТОВ «Завод Емко», "text": "Попередніми місцями роботи Сміяненка є ТОВ «Завод» Емко» — столичний виробник електротехнічної продукції, співвласниками якого є тесть та теща Сміяненка Юрій та Наталія Козубенки."}, {"label": "ТОВ «Торговий дім «Будівельні системи», "text": "Також Сміяненко був директором ТОВ «Торговий дім «Будівельні системи»."}, {"label": "Веб-інтеграція", "text": "У декларації за 2019 Сміяненко задекларував 12,9 тис гривень основного доходу від ТОВ «Веб-інтеграція», де до призначення у САД він був співзасновником."}, {"label": "ТОВ «Торговий дім «Електрокомплект», "text": "Також перед призначенням він вийшов з ТОВ «Торговий дім «Електрокомплект»."}, {"label": "ПП «Аудиторська фірма «Едвайс», "text": "У деклара

With larger prompt and few-shot examples from real data, model become even more unstable. So, next experiment trying to modify baseline prompt to be more strict and concrete.

In [ ]:
train = pd.read_csv("../data/train.csv")

In [73]:
with open("../prompts/ner-prompt-eng.txt", "r", encoding="utf-8") as f:
    template = f.read()

idx = randint(0, len(train) - 1)
response = run_ner(train.text[idx], template)
print(response)

const text = `Малий кивнув і погладив Сусанну по голові — він ніколи не робив цього раніше , то тільки вона , Сусанна , жаліючи , гладила його по голові .

Тож тепер він відчув себе дорослим і сильним .

І несподівано для самого себе заспівав .

Його голосок тремтів , він не був ні чистим , ні красивим .

Він був сильним .

Бо це був голос , що поспішає на допомогу .

Голос , що підтримує і підкріплює .

Голос , що просить сніг прийти .

Голос , що пережив горе .

Справжнє , велике горе .

Голос , що прощався з другом , адже та дівчинка могла би стати його другом .

Та ні , вона вже була його другом .

Раптом Малий відчув , що прабабуся П’ятниця співає разом із ним — всередині нього .

І що голос у неї соковитий і м’який .

Цей голос закутував і огортав , ніс у собі тепло .

Та дівчинка теж заспівала , і Малий усміхнувся до її ляльки .

Він берег тиме цю ляльку .

Тепер це його найрожчий скарб .
Малий співав про любов до тих , що відходять , про пам’ять про них і про зустріч , адже він

In [74]:
y_pred = []

idx = 0
for row in train.head(10).itertuples(index=False):
    y_pred.append({ 'id': row.id, 'entities': run_ner(row.text, template) })
    print("Progress: ", idx)
    idx += 1

Progress:  0
Progress:  1
Progress:  2
Progress:  3
Progress:  4
Progress:  5
Progress:  6
Progress:  7
Progress:  8
Progress:  9


In [75]:
y_pred

[{'id': '582dab45ea473d07323e3c6a0415425c037e63b5a7ab0abd97fb9618799632ec',
  'entities': '[\n  {"label": "MISC", "text": "Помаленько, якось буде."},\n  {"label": "PERIOD", "text": "2014 р."},\n  {"label": "ORG", "text": "Росія"},\n  {"label": "ORG", "text": "Україна"},\n  {"label": "MISC", "text": "Збройні Сили України"},\n  {"label": "PERIOD", "text": "2014 р."},\n  {"label": "ORG", "text": "Росія"},\n  {"label": "MISC", "text": "Німеччина"},\n  {"label": "MISC", "text": "Австралія"},\n  {"label": "MISC", "text": "Швеція"},\n  {"label": "MISC", "text": "США"},\n  {"label": "PERIOD", "text": "2018 р."},\n  {"label": "ORG", "text": "Україна"},\n  {"label": "MISC", "text": "Гаріка Корогодського"}\n]'},
 {'id': 'df0933559918588668cb4219c80908b33cdef041dce468e223030261f1fb2522',
  'entities': '{\n  "label": "MISC",\n  "text": ""\n}\n{\n  "label": "PERS",\n  "text": "Сергій Володимирович"\n}\n{\n  "label": "ORG",\n  "text": "Районна державна адміністрація"\n}\n{\n  "label": "LOC",\n  "text

Now, Results are much better, than previously, from 10 calls most have valid format and can be evaluated. Also, response is faster.

To simplify further work, utils functions, like response validator should be added.

In [53]:
LABELS = ["ART",
"DATE",
"DOC",
"JOB",
"LOC",
"MISC",
"MON",
"ORG",
"PCT",
"PERIOD",
"PERS",
"QUANT",
"TIME"]

In [54]:
import json
import re

def retrieve_json_from_output(output):
    match = re.search(r'\[.*\]', output, flags=re.S)
    if not match:
        return []

    json_text = match.group(0)

    try:
        return json.loads(json_text)
    except:
        pass

    json_text = json_text.replace("'", '"')
    json_text = re.sub(r",\s*}", "}", json_text)
    json_text = re.sub(r",\s*]", "]", json_text)

    try:
        return json.loads(json_text)
    except:
        return []

def filter_entities_by_labels(entities: list, labels: list):
    entities_filtered = []
    missed_values = []
    for entity in entities:
        if entity['label'] in labels:
            entities_filtered.append(entity)
        else:
            missed_values.append(entity)

    return entities_filtered, missed_values

In [76]:
response_filtered = []

for output in y_pred:
    entities = retrieve_json_from_output(output['entities'])
    entities_filtered, missed_values = filter_entities_by_labels(entities, LABELS)
    print("Missed Values: ", len(missed_values), "/", len(entities_filtered))
    response_filtered.append({"entities": entities_filtered, "id": output['id']})

Missed Values:  0 / 14
Missed Values:  0 / 0
Missed Values:  0 / 0
Missed Values:  0 / 0
Missed Values:  0 / 0
Missed Values:  0 / 24
Missed Values:  0 / 13
Missed Values:  0 / 0
Missed Values:  0 / 0
Missed Values:  0 / 0


In [77]:
with open("../prompts/ner-prompt-eng-1.txt", "r", encoding="utf-8") as f:
    template1 = f.read()

y_pred = []

idx = 0
for row in train.sample(10).itertuples(index=False):
    y_pred.append({ 'id': row.id, 'entities': run_ner(row.text, template1) })
    print("Progress: ", idx)
    idx += 1

response_filtered = []

for output in y_pred:
    entities = retrieve_json_from_output(output['entities'])
    entities_filtered, missed_values = filter_entities_by_labels(entities, LABELS)
    print("Missed Values: ", len(missed_values), "/", len(entities_filtered))
    response_filtered.append({"entities": entities_filtered, "id": output['id']})

Progress:  0
Progress:  1
Progress:  2
Progress:  3
Progress:  4
Progress:  5
Progress:  6
Progress:  7
Progress:  8
Progress:  9
Missed Values:  0 / 0
Missed Values:  0 / 0
Missed Values:  0 / 0
Missed Values:  0 / 0
Missed Values:  0 / 12
Missed Values:  0 / 0
Missed Values:  2 / 11
Missed Values:  0 / 0
Missed Values:  0 / 0
Missed Values:  0 / 0


In [79]:
response_filtered

[{'entities': [],
  'id': '43e6f3799b952b78591dc666f072edbad636b4c60dd73f2eb9dbdde8909aa840'},
 {'entities': [],
  'id': 'd97893f330249d3bad8fd63debf63d690677acc7d3d5a7290f2df621fcbb6fba'},
 {'entities': [],
  'id': '301fe85851217df33ca8abaf12c95eb057e2db6229bbefe072d21d10d9c82f83'},
 {'entities': [],
  'id': 'f028885222161fe3b585902d3ab1a6a6eeb0870126cd57859eb60a1cd8dd72da'},
 {'entities': [{'label': 'ORG',
    'text': 'національний природний парк « Зачарований край »'},
   {'label': 'ORG', 'text': 'Всесвітній фонд природи WWF'},
   {'label': 'LOC', 'text': 'Закарпаття'},
   {'label': 'LOC', 'text': 'Гора Бужор'},
   {'label': 'PERIOD', 'text': '2012-2014 роки'},
   {'label': 'PERIOD', 'text': '50-х років минулого століття'},
   {'label': 'PERIOD', 'text': '60-х років минулого століття'},
   {'label': 'ORG', 'text': 'проект із відновлення болота « Чорне багно »'},
   {'label': 'ORG', 'text': 'проект повернення болота до природного стану'},
   {'label': 'PERIOD', 'text': '2013 рік'},
 

As JSON is not the best data format to use with llm, I will try to convert examples and requested answer format to more friendly style. I will use CSV

In [87]:
import json
import csv
from io import StringIO

train = pd.read_csv("../data/train.csv")
train.head()

def json_to_csv_string_manual(json_string):
    data = json.loads(json_string)

    if not data:
        return ""

    # Assuming the JSON is a list of dictionaries, and all dictionaries have the same keys
    # or you want to handle missing keys gracefully.
    fieldnames = list(data[0].keys())

    output = StringIO()
    writer = csv.DictWriter(output, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(data)

    return output.getvalue()

# Example usage:
json_data = train.entities[0]
csv_output = json_to_csv_string_manual(json_data)
print(csv_output)

label,text
MISC,КСВ
MISC,КСВ
MISC,КСВ
MISC,КСВ
ORG,Життєлюб
PERS,Гаріка Корогодського
MISC,Глобальному договору
MISC,Цілям сталого розвитку ООН
ORG,Збройних Сил
MISC,КСВ
ORG,Збройні Сили України
LOC,Криму
LOC,Донбасі
DATE,2014 р .
MON,сотні мільйонів доларів
ORG,ЗСУ
LOC,Україною
ORG,Збройним Силам
DATE,2014 р .
LOC,України
LOC,Росії
ORG,Roshen
LOC,Києвом
DATE,2014 р .
ORG,ЗСУ
MON,700-800 млн грн .
MISC,КСВ
MISC,КСВ
ORG,Фармак
MISC,КСВ
LOC,Німеччини
LOC,Австралії
LOC,Швеції
LOC,США
ORG,Фармак
LOC,Європи
LOC,Азії
DATE,березні 2018 р .
ORG,Фармак
ORG,Екошкола



In [88]:
train['entities_csv'] = train.entities.apply(json_to_csv_string_manual)

In [89]:
with open("../prompts/ner-prompt-csv.txt", "r", encoding="utf-8") as f:
    template_csv = f.read()

y_pred = []

idx = 0
for row in train.sample(10).itertuples(index=False):
    y_pred.append({ 'id': row.id, 'entities': run_ner(row.text, template1) })
    print("Progress: ", idx)
    idx += 1

Progress:  0
Progress:  1
Progress:  2
Progress:  3
Progress:  4
Progress:  5
Progress:  6
Progress:  7
Progress:  8
Progress:  9


In [90]:
y_pred

[{'id': 'a80423172a398a22649b1e3eb5660d8c978816fb98aa7a9c9baedc2a38f6060a',
  'entities': 'Ти можеш використовувати бібліотеку Python з функцією NER для українською мовою. Для цього thou kannst використовувати бібліотеку spaCy.\n\nСпочатку, тобі потрібно завантажити бібліотеку spaCy:\n\n```bash\npip install spacy\npython -m spacy download uk_core_news_sm\n```\n\nЗатем, ти можеш використовувати функцію `ner` для аналізу тексту:\n\n```python\nimport spacy\n\nnlp = spacy.load("uk_core_news_sm")\n\ntext = """\nПриватні підрядники «Харківського метрополітену» отримують гроші за роботи, з якими ще кілька років тому підприємство справлялося самостійно.\nПро це пише Григорій Пирлік на сайті «МедіаПорт».\nКомерційні фірми ремонтують вагони підземки, стежать за ескалаторами, прибирають на станціях.\nЗокрема, вагонами займається ТОВ «Ваго-Рев», ескалаторами – ТОВ «Спеціалізований «Проектно будівельний монтажний поїзд-753», а прибирання метрополітену на себе взяло ТОВ «Компанія «Системи прогресивн

Seems like not a stacking to some specific format is a difficulty for a model. With csv, fraction of wrong answers are the same as with json.

Attempt to breakdown task for two steps - just find named entities. Second call will label it and format in JSON.

In [94]:
with open("../prompts/breakdown-ner-prompt-1.txt", "r", encoding="utf-8") as f:
    template_breakdown= f.read()

y_pred = []

idx = 0
for row in train.sample(10).itertuples(index=False):
    y_pred.append({ 'id': row.id, 'entities': run_ner(row.text, template_breakdown) })
    print("Progress: ", idx)
    idx += 1

Progress:  0
Progress:  1
Progress:  2
Progress:  3
Progress:  4
Progress:  5
Progress:  6
Progress:  7
Progress:  8
Progress:  9


In [112]:
y_pred_ids = [row['id'] for row in y_pred]

for row in train[train['id'].isin(y_pred_ids)].itertuples(index=False):
    print(run_ner(row.text, template_breakdown))


пенсіонер,китайська підробка
Петро,Львові,SoftServe,Дунай,Гюстава Ейфель,Нюґаті,велосипедисти,велодоріжок,велостоянок,місто,кава,шоколад,пані,жіночка,жінка,вік,місто,пагорб,мост,органічна музика,церква,лев,велосипед,єврейська родина,сонце,нічна мерехтлива гірлянда,барвистий шарфик,горошок,керамічні глечики,ґюстава Ейфель,марина,палець,шоколад,пілотська форма,вай-фау,лістя,друзі,безпека
петро,григорій козловський,ірина сердюк,ігорь coronчевський,петро адамик,ігорь зінкевич,андрій гутник,андрій рікота,віталій свішов,іван рудницький,роман Федишин,наталія,іван,STEPAN,Олег Василишин,Андрій Дворакевич,Богданна Зубрицька,Валерій Веремчук,Віталій Свінціцький,Роман Грицевич,Назар Палідович
Потенциал, ТОВ «Науково-практичний центр реконструкції історичного середовища», КOSTянтин РОєнко, КП «Дирекція будівництва шляхово-транспортних споруд м. Києва», ТОВ «Проектне бюро «Юран»
Помаленько, якось буде.
Зевс ЛТД, ДНР, НВФ, Оплот, Кусок, Яшутин, Ігор Васильович, Арс, Ніка, Некст, Зибінов Олександр В'я

### DSPy

In [80]:
import dspy

train, few_shot_records = train_test_split(train, test_size=5, random_state=42, shuffle=True)

In [84]:
train_examples = [dspy.Example(text=train_example.text,
                               entities_json=decode_unicode_escape(train_example.entities))
                  for train_example in few_shot_records.itertuples(index=False)]

In [85]:
from src.promt_automation.ner_module import NERModule
from src.promt_automation.ollama_llama_32 import OllamaLlama32

dspy.configure(lm=OllamaLlama32(model_name="llama3.2"))
optimizer = dspy.BootstrapFewShot()
ner_module = NERModule()

optimized_ner = optimizer.compile(
    student=ner_module,
    trainset=train_examples
)

  0%|          | 0/5 [00:00<?, ?it/s]2025/12/02 18:20:26 ERROR dspy.teleprompt.bootstrap: Failed to run or to evaluate example Example({'text': 'Менш розвинутим промислом , ніж рибальство , було бортництво , яке поступово переросло у пасічництво .\nБортництво – це збирання меду диких бджіл , що роїлись у дуплах дерев на висоті 4-6 м , які називались бортями .\nНа запорозьких вольностях для пасік будували спеціальні зимівники , які розташовувались в урочищах Дніпра , Інгулу та Громоклеї і могли налічувати до 15 тис . вуликів .\nУтримували бджіл у штучно виготовлених вуликах , які називались колоди , дуплянки , сапети .\nСапетові вулики були у козаків найпоширенішими .\nВони плелися із болотяних та плавневих рослин ( очерет , рогоза , верба , солома ) , а в середині обмащувались глиною .\nЗаняття пасічництвом у запорозьких козаків було у великій пошані .\nБагато хто з козаків під кінець життя усамітнювались на пасіці .\nТак , наприклад , кошовий отаман Іван Сірко мав дві пасіки в « Чорно

Bootstrapped 0 full traces after 4 examples for up to 1 rounds, amounting to 5 attempts.
